## Runtime setup (Colab or local)

Run this cell before the analysis. It reuses a local checkout or clones the GitHub repository into Colab, then installs `src/gdsc` and the dependencies declared in `pyproject.toml`. Python 3.12 or later is required.

The clone contains the published default branch; push your changes before opening a fresh Colab runtime. For a private repository, provide an authenticated checkout at `/content/gdsc-project` first. Raw data and local `.env` credentials are not included. Configure COSMIC access in the runtime before downloading expression data. If packages were already imported, restart the runtime after installation and rerun this cell.


In [ ]:
import sys
import subprocess
from pathlib import Path

setup_env_dir = Path.cwd()
if sys.version_info < (3, 12):
    raise RuntimeError("This project requires Python 3.12 or later; select a compatible runtime.")
setup_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "src/gdsc/data.py").is_file() and (p / "pyproject.toml").is_file()), None)
if setup_root is None:
    if "google.colab" not in sys.modules:
        raise RuntimeError("Open this notebook from within the project checkout.")
    setup_root = Path("/content/gdsc-project")
    if not setup_root.exists():
        subprocess.run(["git", "clone", "https://github.com/ajharris/gdsc-project.git", str(setup_root)], check=True)
    if not (setup_root / "src/gdsc/data.py").is_file():
        raise RuntimeError("Expected a project checkout at /content/gdsc-project.")

# Editable installation includes src/gdsc and all declared runtime dependencies.
%pip install -e "$setup_root"
sys.path.insert(0, str(setup_root / "src"))
setup_notebook_dir = setup_root / 'notebooks'
setup_notebook_dir.mkdir(parents=True, exist_ok=True)
%cd $setup_notebook_dir


### Run all: configure credentials once

For unattended Colab setup, open **Secrets** (the key icon), add `COSMIC_AUTHORIZATION` with the value from your local `.env`, and enable **Notebook access**. Then choose **Runtime → Run all**. A signed `COSMIC_LINK` secret is also supported, but expires and may need replacement.

For `.env` setup, place your file in the project root or beside the notebook before Run all. In Colab, upload it to `/content/.env` using the Files panel before Run all. Setup reuses that `.env` or runtime credentials, then checks Colab Secrets. If neither is available, Run all pauses at a file picker for your `.env` and continues after upload. Local `.env` files cannot be read automatically from a remote Colab server. Files selected through the setup picker are removed after loading; secrets are never printed. Saved Colab Secrets can be used again after a runtime restart without another upload.


In [ ]:
# Run before analysis; rerunning also refreshes imported COSMIC configuration.
import sys
import os
import importlib
import tempfile
from pathlib import Path
from dotenv import load_dotenv

def load_runtime_environment():
    def credentials_present():
        return any(os.environ.get(key, "").strip()
                   for key in ("COSMIC_AUTHORIZATION", "COSMIC_LINK"))

    # Local config takes precedence over defaults; existing runtime values
    # and accessible Colab Secrets also work without an upload prompt.
    env_candidates = [setup_root / ".env",
                      globals().get("setup_env_dir", setup_root) / ".env"]
    if "google.colab" in sys.modules:
        env_candidates.append(Path("/content/.env"))
    for env_path in dict.fromkeys(env_candidates):
        if env_path.is_file():
            load_dotenv(env_path, override=True, encoding="utf-8-sig")
            if credentials_present():
                break
    if "google.colab" in sys.modules:
        from google.colab import files, userdata
        if not credentials_present():
            for key in ("COSMIC_AUTHORIZATION", "COSMIC_LINK"):
                try:
                    value = userdata.get(key)
                except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                    continue
                if value and value.strip():
                    os.environ[key] = value.strip()
                    break
        if not credentials_present():
            print("No credentials available. Select your .env once to continue Run all.")
            with tempfile.TemporaryDirectory(prefix="gdsc-env-") as directory:
                env_path = Path(directory) / ".env"
                files.upload_file(str(env_path))
                load_dotenv(env_path, override=True, encoding="utf-8-sig")
    load_dotenv(setup_root / ".env.example", encoding="utf-8-sig")
    authorization_present = bool(os.environ.get("COSMIC_AUTHORIZATION", "").strip())
    link_present = bool(os.environ.get("COSMIC_LINK", "").strip())
    print("COSMIC_AUTHORIZATION:", "set" if authorization_present else "missing or empty")
    print("COSMIC_LINK:", "set" if link_present else "missing or empty")
    if not (authorization_present or link_present):
        raise RuntimeError("No COSMIC credentials loaded. Upload your actual .env, not .env.example; it must contain COSMIC_AUTHORIZATION or COSMIC_LINK.")
    # COSMIC keeps file configuration in module constants. Refresh it if this
    # cell was run after an earlier analysis import in the same runtime.
    if "gdsc.cosmic" in sys.modules:
        importlib.reload(sys.modules["gdsc.cosmic"])
        print("Refreshed imported COSMIC configuration. Rerun subsequent analysis cells.")
    print("Environment loaded. Variable values are not displayed.")

load_runtime_environment()


# LN_IC50 response-metric sensitivity analysis

**Purpose.** Planned sensitivity analysis using LN_IC50 rather than AUC.

**Assumes.** The primary AUC experiment is frozen in [01_main_analysis.ipynb](01_main_analysis.ipynb).

**Produces.** A separate future comparison of performance and feature conclusions. This scaffold contains no implemented or executed sensitivity model.


## Objective

Determine whether primary predictive conclusions are robust when response is represented by LN_IC50. This is not an attempt to improve, reopen, or retune the locked AUC result.


## Relationship to Primary Experiment

The AUC experiment remains frozen. This sensitivity analysis must be reported separately, with its own locked configuration and one final evaluation.


## Fixed Methodology

Reuse GDSC/COSMIC provenance, `lung_NSCLC` cohort, drug-eligibility and deterministic drug-selection rules, COSMIC mapping, target-independent features, grouped splits, training-only preprocessing, validation-only development, and test isolation. Record metric-specific decisions before testing.


## LN_IC50 Response Definition

Specify response direction and validity checks before cohort construction. Do not assume AUC-direction language transfers automatically.


## Cohort Reconstruction

Future work will use `response_metric="LN_IC50"` and report any cohort differences caused by response availability. No cohort is built here.


## Modeling

Future work will fit baselines and candidates using training/validation data only, then lock one strategy. No model fitting or tuning has started.


## Held-out Evaluation

Report test performance only after the LN_IC50 strategy is locked. It must not modify the AUC experiment.


## Feature Comparison

After locking, compare feature findings with fixed AUC results while respecting metric-specific directions.


## Comparison with AUC

Compare cohort composition, validation/test performance, and stable feature patterns; distinguish metric effects from sampling variation.


## Conclusions

No conclusions are available: this is an unexecuted scaffold.
